# Day 8: Feature Engineering & Pipelines

**Dataset:** Titanic cleaned (from Day 2)
**Target:** `survived` (binary)
**Goal:** ColumnTransformer + Pipeline, leakage demonstration, joblib persistence

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report
import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

# Load cleaned data
df = pd.read_csv('../data/titanic_clean.csv')
print(f"Loaded: {df.shape}")

Loaded: (891, 19)


In [2]:
# 1. DEFINE FEATURE TYPES
numeric_features = ['age', 'sibsp', 'parch', 'family_size', 'fare_per_person', 'pclass', 'is_alone']
categorical_features = ['sex', 'embarked', 'deck', 'who', 'age_bin']

X = df[numeric_features + categorical_features]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (712, 12), Test: (179, 12)


In [3]:
# 2. BUILD COLUMTRANSFORMER + PIPELINE (CORRECT WAY)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1))
])

# Cross-validation with pipeline (NO LEAKAGE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipeline_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print("=== Pipeline CV (Correct) ===")
print(f"ROC-AUC: {pipeline_scores.mean():.4f} (+/- {pipeline_scores.std():.4f})")

# Fit on full train, evaluate on test
pipeline.fit(X_train, y_train)
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

print(f"\nTest ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

=== Pipeline CV (Correct) ===
ROC-AUC: 0.8798 (+/- 0.0235)

Test ROC-AUC: 0.8543
              precision    recall  f1-score   support

        Died       0.84      0.83      0.83       110
    Survived       0.73      0.75      0.74        69

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [4]:
# 3. LEAKY APPROACH: Preprocess on FULL data before CV
# This is WRONG but commonly done

# Fit preprocessor on ENTIRE training data (LEAKAGE!)
preprocessor_leaky = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)
preprocessor_leaky.fit(X_train)  # Fits on ALL training data
X_train_leaky = preprocessor_leaky.transform(X_train)
X_test_leaky = preprocessor_leaky.transform(X_test)

# Now CV on already-transformed data (LEAKAGE!)
rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)
leaky_scores = cross_val_score(rf_leaky, X_train_leaky, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print("=== Leaky CV (Wrong) ===")
print(f"ROC-AUC: {leaky_scores.mean():.4f} (+/- {leaky_scores.std():.4f})")
print(f"\nInflation: {leaky_scores.mean() - pipeline_scores.mean():.4f} (optimistic bias)")

=== Leaky CV (Wrong) ===
ROC-AUC: 0.8798 (+/- 0.0238)

Inflation: -0.0000 (optimistic bias)


In [5]:
# 4. ANOTHER LEAKAGE EXAMPLE: Imputer fit on full data
# Manual preprocessing without pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# WRONG: Fit imputer on full X_train
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train[numeric_features])  # LEAKAGE!
X_test_imputed = imputer.transform(X_test[numeric_features])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)  # LEAKAGE!
X_test_scaled = scaler.transform(X_test_imputed)

# Encode categoricals (also leaky if fit on full data)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[categorical_features])  # LEAKAGE!
X_test_cat = encoder.transform(X_test[categorical_features])

X_train_manual = np.hstack([X_train_scaled, X_train_cat])
X_test_manual = np.hstack([X_test_scaled, X_test_cat])

rf_manual = RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1)
manual_scores = cross_val_score(rf_manual, X_train_manual, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print("=== Manual Preprocessing CV (Leaky) ===")
print(f"ROC-AUC: {manual_scores.mean():.4f} (+/- {manual_scores.std():.4f})")
print(f"Inflation vs Pipeline: {manual_scores.mean() - pipeline_scores.mean():.4f}")

=== Manual Preprocessing CV (Leaky) ===
ROC-AUC: 0.8798 (+/- 0.0238)
Inflation vs Pipeline: -0.0000


In [8]:
# 5. FEATURE ENGINEERING INSIDE PIPELINE
# Add custom transformer for feature engineering
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Add derived features: family_size, is_alone, fare_per_person"""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        # X columns: age, sibsp, parch, fare, pclass
        # Assuming column order from numeric_features
        sibsp = X[:, 1]
        parch = X[:, 2]
        fare = X[:, 5]
        pclass = X[:, 6]
        
        family_size = sibsp + parch + 1
        is_alone = (family_size == 1).astype(int)
        fare_per_person = fare / family_size
        
        return np.column_stack([X, family_size, is_alone, fare_per_person])

# Pipeline with feature engineering
pipeline_fe = Pipeline(steps=[
    ('feature_eng', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1))
])

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["family_size"] = X["sibsp"] + X["parch"] + 1
        X["is_alone"] = (X["family_size"] == 1).astype(int)

        # `fare` is not present in X, so retain the existing derived value.
        X["fare_per_person"] = X["fare_per_person"].replace(
            [np.inf, -np.inf], np.nan
        )

        return X


pipeline_fe = Pipeline(steps=[
    ("feature_eng", FeatureEngineer()),
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

fe_scores = cross_val_score(
    pipeline_fe,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("=== Pipeline with Feature Engineering ===")
print(f"ROC-AUC: {fe_scores.mean():.4f} (+/- {fe_scores.std():.4f})")
print(f"Improvement: {fe_scores.mean() - pipeline_scores.mean():.4f}")

=== Pipeline with Feature Engineering ===
ROC-AUC: 0.8798 (+/- 0.0235)
Improvement: 0.0000


In [9]:
# 6. SAVE & LOAD PIPELINE
import joblib
import os

os.makedirs('../models', exist_ok=True)
model_path = '../models/titanic_pipeline.joblib'
joblib.dump(pipeline, model_path)

print(f"Pipeline saved to {model_path}")
print(f"File size: {os.path.getsize(model_path) / 1024:.1f} KB")

# Load and verify
loaded_pipeline = joblib.load(model_path)
y_proba_loaded = loaded_pipeline.predict_proba(X_test)[:, 1]
print(f"Loaded pipeline test ROC-AUC: {roc_auc_score(y_test, y_proba_loaded):.4f}")
print(f"Match: {np.allclose(y_proba, y_proba_loaded)}")

Pipeline saved to ../models/titanic_pipeline.joblib
File size: 817.7 KB
Loaded pipeline test ROC-AUC: 0.8543
Match: True


In [10]:
# 7. PIPELINE INSPECTION: Feature names after one-hot
preprocessor.fit(X_train)
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
all_feature_names = list(numeric_features) + list(cat_feature_names)

print(f"Total features after preprocessing: {len(all_feature_names)}")
print("Feature names:")
for name in all_feature_names:
    print(f"  {name}")

Total features after preprocessing: 28
Feature names:
  age
  sibsp
  parch
  family_size
  fare_per_person
  pclass
  is_alone
  sex_female
  sex_male
  embarked_C
  embarked_Q
  embarked_S
  deck_A
  deck_B
  deck_C
  deck_D
  deck_E
  deck_F
  deck_G
  deck_Unknown
  who_child
  who_man
  who_woman
  age_bin_Adult
  age_bin_Child
  age_bin_Senior
  age_bin_Teen
  age_bin_Young Adult


## Summary Notes

- Pipeline CV score: 
- Leaky CV inflation: 
- Feature engineering improvement: 
- Key leakage lesson: 